In [1]:
import os
import scipy.io
import pandas as pd

INPUT_DIR = "USC-HAD"     # folder where your S01, S02, ... are stored
OUTPUT_DIR = "usc_had_csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def convert_mat_to_csv(mat_path, output_path):
    mat = scipy.io.loadmat(mat_path)

    # Extract the 6-channel sensor readings
    readings = mat["sensor_readings"]

    # Convert readings to DataFrame
    df = pd.DataFrame(
        readings,
        columns=["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    )

    # Add metadata
    df["title"] = mat.get("title", [""])[0]
    df["version"] = mat.get("version", [""])[0]
    df["date"] = mat.get("date", [""])[0]
    df["subject"] = int(mat.get("subject", [[0]])[0][0])
    df["age"] = int(mat.get("age", [[0]])[0][0])
    df["height"] = float(mat.get("height", [[0]])[0][0])
    df["weight"] = float(mat.get("weight", [[0]])[0][0])
    
    df["activity_name"] = mat.get("activity_name", [""])[0]
    df["activity_number"] = int(mat.get("activity_number", [[0]])[0][0])
    df["trial_number"] = int(mat.get("trial_number", [[0]])[0][0])
    
    df["sensor_location"] = mat.get("sensor_location", [""])[0]
    df["sensor_orientation"] = mat.get("sensor_orientation", [""])[0]

    # Save as CSV
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")


# Walk through all subject & activity folders
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".mat"):
            mat_path = os.path.join(root, file)
            csv_name = file.replace(".mat", ".csv")
            output_path = os.path.join(OUTPUT_DIR, csv_name)

            convert_mat_to_csv(mat_path, output_path)


Saved: usc_had_csv\a10t1.csv
Saved: usc_had_csv\a10t2.csv
Saved: usc_had_csv\a10t3.csv
Saved: usc_had_csv\a10t4.csv
Saved: usc_had_csv\a10t5.csv
Saved: usc_had_csv\a11t1.csv
Saved: usc_had_csv\a11t2.csv
Saved: usc_had_csv\a11t3.csv
Saved: usc_had_csv\a11t4.csv
Saved: usc_had_csv\a11t5.csv
Saved: usc_had_csv\a12t1.csv
Saved: usc_had_csv\a12t2.csv
Saved: usc_had_csv\a12t3.csv
Saved: usc_had_csv\a12t4.csv
Saved: usc_had_csv\a12t5.csv
Saved: usc_had_csv\a1t1.csv
Saved: usc_had_csv\a1t2.csv
Saved: usc_had_csv\a1t3.csv
Saved: usc_had_csv\a1t4.csv
Saved: usc_had_csv\a1t5.csv
Saved: usc_had_csv\a2t1.csv
Saved: usc_had_csv\a2t2.csv
Saved: usc_had_csv\a2t3.csv
Saved: usc_had_csv\a2t4.csv
Saved: usc_had_csv\a2t5.csv
Saved: usc_had_csv\a3t1.csv
Saved: usc_had_csv\a3t2.csv
Saved: usc_had_csv\a3t3.csv
Saved: usc_had_csv\a3t4.csv
Saved: usc_had_csv\a3t5.csv
Saved: usc_had_csv\a4t1.csv
Saved: usc_had_csv\a4t2.csv
Saved: usc_had_csv\a4t3.csv
Saved: usc_had_csv\a4t4.csv
Saved: usc_had_csv\a4t5.csv
Saved

In [2]:
# ============================================================
#   USC-HAD FULL PIPELINE
#   1. Load all CSV files
#   2. Segment signals into windows
#   3. Extract features
#   4. Train ML models
#   5. Evaluate performance
# ============================================================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# STEP 1 — Load All CSV Files

CSV_DIR = "usc_had_csv"

all_dfs = []

for file in os.listdir(CSV_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(f"{CSV_DIR}/{file}")
        df["file"] = file
        all_dfs.append(df)

raw_df = pd.concat(all_dfs, ignore_index=True)
print("Loaded dataset:", raw_df.shape)
print(raw_df.head())


# STEP 2 — Segment Each Trial Into Windows

def segment_signal(df, window=128, step=64):
    segments = []
    labels = []

    data = df[['acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z']].values
    label = df['activity_number'].iloc[0]

    for start in range(0, len(data)-window, step):
        segment = data[start:start+window]
        segments.append(segment)
        labels.append(label)

    return segments, labels


# Collect segments for ALL files
all_segments = []
all_labels = []

for file in os.listdir(CSV_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(f"{CSV_DIR}/{file}")
        segs, labs = segment_signal(df)
        all_segments.extend(segs)
        all_labels.extend(labs)

print("Total segments:", len(all_segments))


# STEP 3 — Feature Extraction

def extract_features(segment):
    segment = np.array(segment)

    feats = {}
    cols = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]

    for i, col in enumerate(cols):
        data = segment[:, i]
        feats[f"{col}_mean"] = data.mean()
        feats[f"{col}_std"] = data.std()
        feats[f"{col}_min"] = data.min()
        feats[f"{col}_max"] = data.max()
        feats[f"{col}_var"] = data.var()

    return feats
    
#feature dataset building
feature_rows = []
for seg, label in zip(all_segments, all_labels):
    f = extract_features(seg)
    f["label"] = label
    feature_rows.append(f)

features_df = pd.DataFrame(feature_rows)

print("Feature dataset:", features_df.shape)
print(features_df.head())

# STEP 4 — Train-Test Split + Standardization

X = features_df.drop("label", axis=1)
y = features_df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


# STEP 5 — Train Multiple ML Models

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200),
    "SVM (RBF)": SVC(kernel="rbf"),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier()
}

# STEP 6 — Evaluate All Models

for name, model in models.items():
    print("\n==============================")
    print(f"Training: {name}")
    print("==============================")

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print(classification_report(y_test, preds))


Loaded dataset: (260767, 19)
      acc_x     acc_y    acc_z    gyro_x    gyro_y    gyro_z  \
0  0.476021 -0.457104  0.67416  0.276908  0.145721  0.047719   
1  0.479454 -0.460723  0.67416 -0.125527 -0.256299 -0.353992   
2  0.479454 -0.460723  0.67416  0.276430 -0.657050 -0.352874   
3  0.479454 -0.464342  0.67416 -0.126003 -0.253416  0.049800   
4  0.479454 -0.457104  0.67416 -0.125605  0.148945  0.451203   

                                  title  version       date  subject  age  \
0  USC Human Activity Dataset (USC-HAD)      1.0  June-2012        9    3   
1  USC Human Activity Dataset (USC-HAD)      1.0  June-2012        9    3   
2  USC Human Activity Dataset (USC-HAD)      1.0  June-2012        9    3   
3  USC Human Activity Dataset (USC-HAD)      1.0  June-2012        9    3   
4  USC Human Activity Dataset (USC-HAD)      1.0  June-2012        9    3   

   height  weight  activity_name  activity_number  trial_number  \
0     1.0     6.0            NaN                1       

C:\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Python310\lib\subprocess.py", line 969, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Python310\lib\subprocess.py", line 1438, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


              precision    recall  f1-score   support

           1       0.97      1.00      0.99       327
           2       1.00      0.89      0.94        74
           3       1.00      0.98      0.99        58
           4       0.98      0.99      0.98        97
           5       1.00      0.98      0.99        85
           6       1.00      1.00      1.00        50
           7       1.00      1.00      1.00        21
           8       1.00      1.00      1.00        37
           9       0.98      1.00      0.99        48

    accuracy                           0.98       797
   macro avg       0.99      0.98      0.99       797
weighted avg       0.99      0.98      0.98       797


Training: Decision Tree
              precision    recall  f1-score   support

           1       0.98      0.99      0.98       327
           2       0.97      0.86      0.91        74
           3       0.92      1.00      0.96        58
           4       0.97      0.99      0.98        97

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

X = features_df.drop(columns=["label"]).values
y_true = features_df["label"].values   

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# STEP 2 — Build Autoencoder (unsupervised)

input_dim = X_scaled.shape[1]

inputs = Input(shape=(input_dim,))
encoded = Dense(32, activation='relu')(inputs)
encoded = Dense(16, activation='relu')(encoded)

decoded = Dense(32, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs, decoded)
autoencoder.compile(optimizer=Adam(0.001), loss='mse')

# STEP 3 — Train Autoencoder on ALL data

history = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)


#reconstruction error
X_recon = autoencoder.predict(X_scaled)
recon_error = np.mean((X_scaled - X_recon)**2, axis=1)

# STEP 5 — Choose threshold to mark anomalies

threshold = np.percentile(recon_error, 95)
print("Threshold selected:", threshold)

# Predict anomalies
y_pred = (recon_error > threshold).astype(int)


# Convert activity labels to anomaly ground truth for evaluation:
# We treat RARE activities as anomalies automatically (unsupervised evaluation)
# Any activity with low frequency in dataset = anomaly
# Example: mark top 70% frequent = normal, bottom 30% = anomaly

from collections import Counter

activity_counts = Counter(y_true)
sorted_counts = sorted(activity_counts.items(), key=lambda x: x[1], reverse=True)

# Mark 70% frequent activities as normal
top_70_percent = int(len(sorted_counts) * 0.7)
normal_activities = [a for a, _ in sorted_counts[:top_70_percent]]
anomaly_activities = [a for a, _ in sorted_counts[top_70_percent:]]

def map_eval(x):
    return 0 if x in normal_activities else 1

y_eval = np.array([map_eval(x) for x in y_true])

print("\nEvaluation based on activity frequency:")
print("Normal:", normal_activities)
print("Anomaly:", anomaly_activities)

print("\nAccuracy:", accuracy_score(y_eval, y_pred))
print("\nClassification Report:")
print(classification_report(y_eval, y_pred, target_names=["Normal","Anomaly"]))


Epoch 1/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.9615 - val_loss: 0.2258
Epoch 2/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3110 - val_loss: 0.1053
Epoch 3/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1479 - val_loss: 0.0833
Epoch 4/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1028 - val_loss: 0.0836
Epoch 5/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0828 - val_loss: 0.0796
Epoch 6/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0705 - val_loss: 0.0740
Epoch 7/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0576 - val_loss: 0.0713
Epoch 8/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0509 - val_loss: 0.0669
Epoch 9/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0493 - val_loss: 0.0632
Epoch 10/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0418 - val_loss: 0.0611
Epoch 11/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0400 - val_loss: 0.0570
Epoch 12/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0366 - val_lo

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

df = features_df.copy()

X = df.drop(columns=["label"]).values
y = df["label"].values

sc = StandardScaler()
X = sc.fit_transform(X)

le = LabelEncoder()
y = le.fit_transform(y)

A = kneighbors_graph(X, n_neighbors=10, mode='connectivity', include_self=False)
A = A.toarray()
edge_index = np.vstack(np.nonzero(A))

x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)
edge_index = torch.tensor(edge_index, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)
train_idx = torch.tensor(train_idx, dtype=torch.long)
test_idx = torch.tensor(test_idx, dtype=torch.long)

class GCN(nn.Module):
    def __init__(self, f_in, f_hid, f_out):
        super().__init__()
        self.c1 = GCNConv(f_in, f_hid)
        self.c2 = GCNConv(f_hid, f_out)
    def forward(self, x, edge_index):
        x = self.c1(x, edge_index)
        x = F.relu(x)
        x = self.c2(x, edge_index)
        return x

model = GCN(X.shape[1], 64, len(np.unique(y)))
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(40):
    opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[train_idx], y[train_idx])
    loss.backward()
    opt.step()

pred = out[test_idx].argmax(1)
acc = (pred == y[test_idx]).float().mean().item()
print("Accuracy:", acc)


Accuracy: 0.9786700010299683


In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, accuracy_score

DATA_FOLDER = "usc_had_csv"
WINDOW = 100
STEP = 50

X_list, y_list = [], []

for file in os.listdir(DATA_FOLDER):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_FOLDER, file))
        label = int(df["activity_number"].iloc[0])
        df = df[["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]].values
        
        for i in range(0, len(df) - WINDOW, STEP):
            segment = df[i:i+WINDOW]
            X_list.append(segment)
            y_list.append(label)

X = np.array(X_list)
y = np.array(y_list)

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_cat = to_categorical(y_enc)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, shuffle=True)

model = Sequential()
model.add(Conv1D(64, 3, activation='relu', input_shape=(WINDOW, 6)))
model.add(MaxPooling1D(2))
model.add(Conv1D(128, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(LSTM(128, return_sequences=False))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dense(y_cat.shape[1], activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.1)

y_pred = model.predict(X_test)
y_pred_cls = np.argmax(y_pred, axis=1)
y_test_cls = np.argmax(y_test, axis=1)

acc = accuracy_score(y_test_cls, y_pred_cls)
print("Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_test_cls, y_pred_cls))


C:\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.5323 - loss: 1.4945 - val_accuracy: 0.8652 - val_loss: 0.3781
Epoch 2/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8729 - loss: 0.3964 - val_accuracy: 0.9118 - val_loss: 0.2326
Epoch 3/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8918 - loss: 0.2853 - val_accuracy: 0.9289 - val_loss: 0.1672
Epoch 4/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9354 - loss: 0.1839 - val_accuracy: 0.9779 - val_loss: 0.0807
Epoch 5/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9680 - loss: 0.1178 - val_accuracy: 0.9559 - val_loss: 0.1445
Epoch 6/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9678 - loss: 0.1116 - val_accuracy: 0.9804 - val_loss: 0.0682
Epoch 7/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9763 - loss: 0.0803 - val_accuracy: 0.9902 - val_loss: 0.0547
Epoch 8/20
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9780 - loss: 0.0653 - val_accuracy: 0.9853 - v

In [7]:
print("\nChecking NULL values in features_df:")
print(features_df.isnull().sum())
print("Any NULLs? ->", features_df.isnull().values.any())



Checking NULL values in features_df:
acc_x_mean     0
acc_x_std      0
acc_x_min      0
acc_x_max      0
acc_x_var      0
acc_y_mean     0
acc_y_std      0
acc_y_min      0
acc_y_max      0
acc_y_var      0
acc_z_mean     0
acc_z_std      0
acc_z_min      0
acc_z_max      0
acc_z_var      0
gyro_x_mean    0
gyro_x_std     0
gyro_x_min     0
gyro_x_max     0
gyro_x_var     0
gyro_y_mean    0
gyro_y_std     0
gyro_y_min     0
gyro_y_max     0
gyro_y_var     0
gyro_z_mean    0
gyro_z_std     0
gyro_z_min     0
gyro_z_max     0
gyro_z_var     0
label          0
dtype: int64
Any NULLs? -> False


In [10]:
print(len(features_df))

3984
